# Liu-like sepsis pipeline

This notebook runs the full modular pipeline for:
- GLM
- XGBoost (if available)
- GRU

It also computes patient-time early warning metrics and saves outputs.

In [2]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.notebook import tqdm

from config import ExperimentConfig
from data import DataModule
from evaluation import EvaluationModule
from models.glm_model import LiuLikeGLM
from models.xgb_model import LiuLikeXGBoost
from models.gru_model import LiuLikeGRU

warnings.filterwarnings("ignore")

print("cwd:", Path.cwd())
print("files:", sorted([p.name for p in Path(".").iterdir()]))

cwd: /Users/roseva1/Desktop/home/rose1838/SP26/PUBH8475/PUBH8475/Final
files: ['.DS_Store', 'README.md', '__pycache__', 'config.py', 'data', 'data.py', 'draft', 'evaluation.py', 'models', 'results_modular', 'results_notebook', 'run_experiment.py', 'run_models.ipynb']


In [3]:
cfg = ExperimentConfig()
cfg.data_dir = Path("./data")
cfg.output_dir = Path("./results_notebook")
cfg.ensure_output_dir()

dm = DataModule(cfg)
ev = EvaluationModule()

dm.set_global_seed(cfg.seed)

print("DATA_DIR:", cfg.data_dir.resolve() if cfg.data_dir.exists() else cfg.data_dir)
print("OUTPUT_DIR:", cfg.output_dir.resolve())
print("seed:", cfg.seed)

DATA_DIR: /Users/roseva1/Desktop/home/rose1838/SP26/PUBH8475/PUBH8475/Final/data
OUTPUT_DIR: /Users/roseva1/Desktop/home/rose1838/SP26/PUBH8475/PUBH8475/Final/results_notebook
seed: 1


## Load and prepare data

In [4]:
top_bar = tqdm(total=4, desc="Notebook pipeline", position=0, leave=True)

print("Loading all patients...")
df, summaries = dm.load_all_patients()
top_bar.update(1)

print("Splitting patients...")
train_ids, test_ids = dm.split_patients(summaries, test_size=cfg.test_size, seed=cfg.seed)
top_bar.update(1)

print("Engineering features...")
df_feat = dm.add_engineered_features(df)
top_bar.update(1)

print("Building static dataset...")
static_ds = dm.build_static_dataset(df_feat, train_ids, test_ids)
top_bar.update(1)

top_bar.close()

summary_preview = pd.DataFrame([s.__dict__ for s in summaries])
display(summary_preview.head())

print("Full data shape:", df.shape)
print("Feature-engineered shape:", df_feat.shape)
print("Train patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Number of modeling features:", len(static_ds.feature_cols))
print("First 10 features:", static_ds.feature_cols[:10])
print("Static X_train:", static_ds.X_train.shape, "positives:", int(np.sum(static_ds.y_train)))
print("Static X_test:", static_ds.X_test.shape, "positives:", int(np.sum(static_ds.y_test)))

Notebook pipeline:   0%|          | 0/4 [00:00<?, ?it/s]

Loading all patients...
Splitting patients...
Engineering features...
Building static dataset...


,patient_id,n_rows,label_start_idx,became_positive
0,p000001,54,NaN,False
1,p000002,23,NaN,False
2,p000003,48,NaN,False
3,p000004,29,NaN,False
4,p000005,48,NaN,False


Full data shape: (1552210, 43)
Feature-engineered shape: (1552210, 123)
Train patients: 28235
Test patients: 12101
Number of modeling features: 120
First 10 features: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3']
Static X_train: (969736, 120) positives: 4090
Static X_test: (415908, 120) positives: 1754


## GLM

In [ ]:
glm_bar = tqdm(total=3, desc="GLM", leave=True)

glm = LiuLikeGLM(random_state=cfg.seed)
glm.fit(static_ds.X_train, static_ds.y_train)
glm_bar.update(1)

glm_probs = glm.predict_proba(static_ds.X_test)
glm_metrics = ev.evaluate_binary_probabilities(static_ds.y_test, glm_probs)
glm_bar.update(1)

glm_scores = ev.build_full_patient_time_scores_static(
    glm,
    df_feat,
    test_ids,
    static_ds.feature_cols
)

glm_ewt = ev.compute_early_warning_metrics(
    glm_scores,
    glm_metrics.optimal_threshold,
    cfg.label_lead_hours
)
glm_bar.update(1)
glm_bar.close()

glm_scores.to_csv(cfg.output_dir / "glm_test_time_scores.csv", index=False)

print("GLM row-level metrics")
display(pd.DataFrame([glm_metrics.__dict__]))
print("GLM patient-time metrics")
display(pd.DataFrame([glm_ewt]))

GLM:   0%|          | 0/3 [00:00<?, ?it/s]

## XGBoost

In [ ]:
xgb = None
xgb_metrics = None
xgb_ewt = None
xgb_scores = None

try:
    xgb_bar = tqdm(total=3, desc="XGBoost", leave=True)
    xgb = LiuLikeXGBoost(random_state=cfg.seed)
    xgb.fit(static_ds.X_train, static_ds.y_train)
    xgb_bar.update(1)

    xgb_probs = xgb.predict_proba(static_ds.X_test)
    xgb_metrics = ev.evaluate_binary_probabilities(static_ds.y_test, xgb_probs)
    xgb_bar.update(1)

    xgb_scores = ev.build_full_patient_time_scores_static(
        xgb,
        df_feat,
        test_ids,
        static_ds.feature_cols
    )

    xgb_ewt = ev.compute_early_warning_metrics(
        xgb_scores,
        xgb_metrics.optimal_threshold,
        cfg.label_lead_hours
    )
    xgb_bar.update(1)
    xgb_bar.close()

    xgb_scores.to_csv(cfg.output_dir / "xgb_test_time_scores.csv", index=False)

    print("XGBoost row-level metrics")
    display(pd.DataFrame([xgb_metrics.__dict__]))
    print("XGBoost patient-time metrics")
    display(pd.DataFrame([xgb_ewt]))

except Exception as e:
    print("XGBoost skipped:", repr(e))

## GRU

In [ ]:
print("Building GRU sequences...")
seq_bar = tqdm(total=2, desc="GRU sequence prep", leave=True)

train_seq_X, train_seq_y = dm.build_patient_sequences(
    df_feat,
    train_ids,
    static_ds.feature_cols
)
seq_bar.update(1)

test_seq_X, test_seq_y = dm.build_patient_sequences(
    df_feat,
    test_ids,
    static_ds.feature_cols
)
seq_bar.update(1)
seq_bar.close()

print("Train sequences:", train_seq_X.shape, "positives:", int(np.sum(train_seq_y)))
print("Test sequences:", test_seq_X.shape, "positives:", int(np.sum(test_seq_y)))

In [ ]:
gru_bar = tqdm(total=3, desc="GRU", leave=True)

gru = LiuLikeGRU(
    random_state=cfg.seed,
    hidden_size=cfg.rnn_hidden_size,
    num_layers=cfg.rnn_num_layers,
    dropout=cfg.rnn_dropout,
    epochs=cfg.epochs,
    batch_size=cfg.batch_size,
    learning_rate=cfg.learning_rate,
)

gru.fit(train_seq_X, train_seq_y)
gru_bar.update(1)

gru_probs = gru.predict_proba(test_seq_X)
gru_metrics = ev.evaluate_binary_probabilities(test_seq_y, gru_probs)
gru_bar.update(1)

gru_scores = ev.build_full_patient_time_scores_rnn(
    gru,
    df_feat,
    test_ids,
    static_ds.feature_cols,
    seq_len=cfg.seq_len
)

gru_ewt = ev.compute_early_warning_metrics(
    gru_scores,
    gru_metrics.optimal_threshold,
    cfg.label_lead_hours
)
gru_bar.update(1)
gru_bar.close()

gru_scores.to_csv(cfg.output_dir / "gru_test_time_scores.csv", index=False)

print("GRU row-level metrics")
display(pd.DataFrame([gru_metrics.__dict__]))
print("GRU patient-time metrics")
display(pd.DataFrame([gru_ewt]))

## Save metrics and summary

In [ ]:
results = {
    "glm_row": glm_metrics.__dict__,
    "glm_patient_time": glm_ewt,
    "gru_row": gru_metrics.__dict__,
    "gru_patient_time": gru_ewt,
}

if xgb_metrics is not None and xgb_ewt is not None:
    results["xgb_row"] = xgb_metrics.__dict__
    results["xgb_patient_time"] = xgb_ewt

with open(cfg.output_dir / "metrics.json", "w") as f:
    json.dump(results, f, indent=2)

with open(cfg.output_dir / "patient_split.json", "w") as f:
    json.dump({
        "train_patient_ids": train_ids,
        "test_patient_ids": test_ids
    }, f, indent=2)

summary_rows = [
    {
        "model": "GLM",
        "AUC": results["glm_row"]["auc"],
        "Sensitivity": results["glm_row"]["sensitivity_at_optimal_threshold"],
        "Specificity": results["glm_row"]["specificity_at_optimal_threshold"],
        "Median Early Warning (hrs)": results["glm_patient_time"]["median_early_warning_time_hours"],
    },
    {
        "model": "GRU",
        "AUC": results["gru_row"]["auc"],
        "Sensitivity": results["gru_row"]["sensitivity_at_optimal_threshold"],
        "Specificity": results["gru_row"]["specificity_at_optimal_threshold"],
        "Median Early Warning (hrs)": results["gru_patient_time"]["median_early_warning_time_hours"],
    },
]

if "xgb_row" in results:
    summary_rows.insert(1, {
        "model": "XGBoost",
        "AUC": results["xgb_row"]["auc"],
        "Sensitivity": results["xgb_row"]["sensitivity_at_optimal_threshold"],
        "Specificity": results["xgb_row"]["specificity_at_optimal_threshold"],
        "Median Early Warning (hrs)": results["xgb_patient_time"]["median_early_warning_time_hours"],
    })

summary_df = pd.DataFrame(summary_rows)

print("Saved outputs to:", cfg.output_dir.resolve())
display(pd.DataFrame(results).T)
display(summary_df)